# 02 · Panels and walk-forward forecasts (design Sections 4-5)

Builds the panels (cleaning: bad prints that reverse next day, forward-fill ≤ 5 days) and runs the walk-forward
forecasts for every panel and horizon: DLinear-QLIKE (ours, 5 seeds), LSTM (base-paper style), HAR (per asset),
GARCH(1,1)-t, EWMA, naive, and quantile-HAR for the CQR comparator. Expanding window, refits every year (assets) or
every 5 / 2 years (US / international factors), purged and embargoed inner validation.

**Use a GPU runtime.** Finished panel-horizon pairs are saved in `results/<mode>/predictions/` and loaded on re-run;
delete a file to recompute it. Approximate T4 times: assets h=21 20-30 min, h=5 30-45 min, h=1 1-2 h; factor panels
10-20 min each. You can restrict `JOBS` below to run in pieces.

In [ ]:
# ---------------------------------------------------------------- setup (run this cell first)
import os, sys, subprocess, warnings
warnings.filterwarnings("ignore")
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = "/content/drive/MyDrive/CalVol_project"   # <- change if you placed the folder elsewhere
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{PROJECT_DIR}/requirements.txt"], check=True)
else:
    PROJECT_DIR = os.path.abspath("..")
os.chdir(PROJECT_DIR)
sys.path.insert(0, os.path.join(PROJECT_DIR, "src"))

import json
import numpy as np, pandas as pd
from IPython.display import display, Image, Markdown
from calvol.utils import load_config, get_paths, check_gate
from calvol import pipeline as pp
pd.set_option("display.max_columns", 40); pd.set_option("display.width", 220)

cfg = load_config(PROJECT_DIR)          # mode / sign-off / smoke test are set in config/config.yaml (project: ...)
MODE = check_gate(cfg)                  # refuses a holdout run without sign-off and a frozen commit
P = get_paths(cfg)
PER = pp.eval_split(cfg)
print(f"MODE = {MODE.upper()}  |  evaluation period = {PER}  |  smoke test = {cfg['project']['smoke_test']}")
print("Results folder:", P.results)

def show(name):
    f = P.figures / f"{name}.png"
    if f.exists():
        display(Image(filename=str(f), width=760))
    else:
        print("(figure not produced:", name, ")")

def table(name, round_=4):
    f = P.tables / f"{name}.csv"
    if f.exists():
        display(pd.read_csv(f).round(round_))
    else:
        print("(table not produced:", name, ")")

In [ ]:
out = pp.stage_panels(cfg)
print(json.dumps(out, indent=1, default=str))
table("assets_instruments", 3)

## Forecasts

In [ ]:
JOBS = [("assets", h) for h in cfg["panels"]["assets"]["horizons"]] + [(p, 21) for p in pp.FACTOR_PANELS]
for name, h in JOBS:
    pred = pp.stage_forecasts(cfg, name, h)
    print(name, h, pred.shape, pred["split"].value_counts().to_dict())

## Forecast accuracy (H4; QLIKE, MSE, Mincer-Zarnowitz, Diebold-Mariano, Model Confidence Set)
DM columns: positive = the row model has lower QLIKE than the reference.

In [ ]:
for name, h in JOBS:
    display(Markdown(f"### {name}, h = {h}"))
    display(pp.forecast_tables(cfg, name, h).round(4))

## Selected DLinear hyper-parameters per refit (appendix)

In [ ]:
for name, h in JOBS:
    g = P.tables / f"{name}_h{h}_garch_fallback.csv"
    if g.exists():
        display(Markdown(f"GARCH status by year, {name} h={h}")); display(pd.read_csv(g).round(3))
    f = P.tables / f"{name}_h{h}_hyperparams.csv"
    if f.exists():
        display(Markdown(f"**{name}, h = {h}**")); display(pd.read_csv(f))